# Trimming Assignmemnt Files

# Notebook overview

This notebook records the **text-trimming and sample-construction pipeline** used to prepare the writing and coding corpora for the study. It is organized mostly by institution, cohort, or processing round. The notebook keeps the chronological development of the pipeline, so some later blocks refine or extend earlier ones rather than replacing them.

**Main workflow**
1. Trim and prepare assignment files by institution and subgroup.
2. Extract code links and candidate code files for the coding-analysis branch.
3. Run later trimming rounds and quality-control statistics.
4. Build the pre-Fall-2022 AI-free sample.
5. Resolve remaining issues such as tables of contents, very short files, and oversized subfolders.

# Taibah Assignments

## Taibah/First Year

### Process Taibah/First Year

This block runs the processing pipeline for **Taibah/First Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re

import pdfplumber
from openai import OpenAI

PDF_ROOT = os.environ["PDF_ROOT"]
OUT_ROOT = os.environ["OUT_ROOT"]
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
MAX_CHARS_PER_CHUNK = 6000
MAX_OUTPUT_TOKENS = int(os.environ.get("MAX_OUTPUT_TOKENS", "4096"))

client = OpenAI()
os.makedirs(OUT_ROOT, exist_ok=True)


def extract_pdf_text(pdf_path: str) -> str:
    """Extract text page by page while preserving page separation."""
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = (
                    (page.extract_text() or "")
                    .replace("\r\n", "\n")
                    .replace("\r", "\n")
                )
                parts.append(text)
    except Exception as exc:
        print(f"[PDF ERROR] {pdf_path}: {exc}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(
    text: str, max_chars: int = MAX_CHARS_PER_CHUNK
) -> list[str]:
    """Split text near paragraph boundaries to limit model input size."""
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    instructions = """
Select and keep only text suitable for AI-writing detection.

Keep verbatim:
- Complete student-written paragraphs, including explanations, arguments, reflections, and descriptions.
- Bulleted or numbered points with substantive content.
- Headings associated with kept content.

Remove:
- Institutional front matter, templates, course information, student information, and repeated headers or footers.
- Tables of contents, indexes, lists of figures or tables, symbol lists, abbreviations, acknowledgements, copyright pages, appendices, references, and bibliographies.
- Table and figure captions, table content, and figure labels.
- Equations, formula-heavy lines, and text dominated by numbers or symbols.
- Page numbers, PDF artifacts, and other non-prose content.

Do not paraphrase, summarize, or alter wording. Preserve the original order. You may join broken lines within paragraphs. Keep headings on their own lines and separate paragraphs with blank lines. Output only the cleaned text.
"""
    try:
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=chunk_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            temperature=0.1,
        )
        return response.output_text
    except Exception as exc:
        print(f"[API ERROR] {exc}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)} to {MODEL}...")
        cleaned = call_ai_trimmer(chunk).strip()
        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n\n".join(cleaned_parts)


total_files = 0
records = []

for root, _, files in os.walk(PDF_ROOT):
    for filename in files:
        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(root, filename)
        relative_path = os.path.relpath(pdf_path, PDF_ROOT)
        relative_dir = os.path.dirname(relative_path)
        base_name = os.path.splitext(filename)[0]

        output_dir = os.path.join(OUT_ROOT, relative_dir)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"{base_name}_ai_trimmed.txt")

        print(f"\n=== Processing: {relative_path} ===")
        raw_text = extract_pdf_text(pdf_path)
        raw_length = len(raw_text)

        if not raw_text:
            print("  [WARN] No text extracted, skipping.")
            continue

        cleaned_text = ai_trim_whole_document(raw_text)
        kept_length = len(cleaned_text)
        percent_kept = 100.0 * kept_length / raw_length

        with open(output_path, "w", encoding="utf-8") as output_file:
            output_file.write(cleaned_text)

        records.append((relative_path, raw_length, kept_length, percent_kept))
        total_files += 1
        print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}% of characters)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")
for relative_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{relative_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

Taibah — Second Year

### Process Taibah/Second Year

This block runs the processing pipeline for **Taibah/Second Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

# Jeddah Assignments

## Jeddah/First Year

### Process Jeddah/First Year

This block runs the processing pipeline for **Jeddah/First Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re

import pdfplumber
from openai import OpenAI

PDF_ROOT = os.environ["PDF_ROOT"]
OUT_ROOT = os.environ["TRIMMED_TEXT_ROOT"]
MAX_CHARS_PER_CHUNK = 6000

client = OpenAI()


def extract_pdf_text(pdf_path: str) -> str:
    """Extract text page by page while retaining document order."""
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = (
                    (page.extract_text() or "")
                    .replace("\r\n", "\n")
                    .replace("\r", "\n")
                )
                parts.append(text)
    except Exception as exc:
        print(f"[PDF ERROR] {pdf_path}: {exc}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(
    text: str, max_chars: int = MAX_CHARS_PER_CHUNK
) -> list[str]:
    """Keep paragraphs together where possible to preserve local context."""
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    instructions = """
Select and retain only text suitable for AI-writing detection.

Keep verbatim:
- Complete paragraphs that appear to be student-authored prose, including explanations, arguments, reflections, and descriptions.
- Bulleted or numbered lists with substantive content.
- Section headings associated with retained content.

Remove:
- Institutional headers, course information, student details, project templates, dates, and repeated boilerplate.
- Tables of contents, indexes, lists of figures or tables, symbol lists, abbreviations, acknowledgements, copyright pages, appendices, references, and bibliographies.
- Table and figure captions, table content, figure labels, equations, formula-heavy lines, and text dominated by numbers or symbols.
- Page numbers, footers, PDF extraction artifacts, and repeated non-prose content.

Do not paraphrase, summarize, or alter retained wording. Preserve the original order. You may join broken lines within a paragraph. Keep headings on their own lines and separate paragraphs with blank lines.

Output only the cleaned text.
"""
    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            instructions=instructions,
            input=chunk_text,
            max_output_tokens=8192,
            temperature=0.1,
        )
        return response.output_text
    except Exception as exc:
        print(f"[API ERROR] {exc}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)} to GPT-4.1-mini...")
        cleaned = call_ai_trimmer(chunk).strip()
        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n\n".join(cleaned_parts)


total_files = 0
records = []

for root, _dirs, files in os.walk(PDF_ROOT):
    for filename in files:
        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(root, filename)
        relative_path = os.path.relpath(pdf_path, PDF_ROOT)
        relative_dir = os.path.dirname(relative_path)
        base_name = os.path.splitext(filename)[0]

        output_dir = os.path.join(OUT_ROOT, relative_dir)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"{base_name}_ai_trimmed.txt")

        print(f"\n=== Processing: {relative_path} ===")
        raw_text = extract_pdf_text(pdf_path)
        raw_length = len(raw_text)

        if not raw_text:
            print("  [WARN] No text extracted; skipping.")
            continue

        cleaned_text = ai_trim_whole_document(raw_text)
        kept_length = len(cleaned_text)
        percent_kept = 100.0 * kept_length / raw_length

        with open(output_path, "w", encoding="utf-8") as output_file:
            output_file.write(cleaned_text)

        records.append((relative_path, raw_length, kept_length, percent_kept))
        total_files += 1
        print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}% of characters)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")
for relative_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{relative_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

## Jeddah/Second Year

### Process Jeddah/Second Year

This block runs the processing pipeline for **Jeddah/Second Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

# KFUPM Written Assignments

## KFUPM/First Semester/KFUPM-2

### Process KFUPM/First Semester/KFUPM-2

This block runs the processing pipeline for **KFUPM/First Semester/KFUPM-2**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re

import pdfplumber
from openai import OpenAI

PDF_ROOT = os.environ["PDF_ROOT"]
OUT_ROOT = os.environ["OUT_ROOT"]
MAX_CHARS_PER_CHUNK = 6_000
MAX_OUTPUT_TOKENS = 32_768

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this script.")

os.makedirs(OUT_ROOT, exist_ok=True)
client = OpenAI()


def extract_pdf_text(pdf_path: str) -> str:
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = (
                    (page.extract_text() or "")
                    .replace("\r\n", "\n")
                    .replace("\r", "\n")
                )
                parts.append(text)
    except Exception as exc:
        print(f"[PDF ERROR] {pdf_path}: {exc}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(
    text: str, max_chars: int = MAX_CHARS_PER_CHUNK
) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    # The model may delete text but must not rewrite retained student prose.
    instructions = """
Select and retain only student-authored prose suitable for AI-writing detection.

Keep verbatim:
- Complete explanatory, argumentative, reflective, or descriptive paragraphs.
- Bulleted or numbered points containing substantive writing.
- Headings associated with retained content.

Remove:
- Institutional headers, course metadata, student identifiers, dates, repeated headers, and footers.
- Tables of contents, indexes, lists of figures or tables, acknowledgements, copyright pages,
  appendices, references, and bibliographies.
- Tables, figures, captions, labels, equations, formula-heavy text, and lines dominated by numbers
  or symbols.
- Page numbers, PDF extraction artifacts, and other non-prose boilerplate.

Do not paraphrase, summarize, add, or reorder text. Broken lines within a retained paragraph may be
joined. Keep headings on separate lines and separate paragraphs with blank lines. Return only the
cleaned text.
"""
    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            instructions=instructions,
            input=chunk_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            temperature=0.1,
        )
        return response.output_text
    except Exception as exc:
        print(f"[API ERROR] {exc}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)} to GPT-4.1-mini...")
        cleaned = call_ai_trimmer(chunk).strip()
        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n\n".join(cleaned_parts)


total_files = 0
records = []

for root, _dirs, files in os.walk(PDF_ROOT):
    for filename in files:
        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(root, filename)
        relative_path = os.path.relpath(pdf_path, PDF_ROOT)
        relative_dir = os.path.dirname(relative_path)
        output_dir = os.path.join(OUT_ROOT, relative_dir)
        output_name = f"{os.path.splitext(filename)[0]}_ai_trimmed.txt"
        output_path = os.path.join(output_dir, output_name)

        os.makedirs(output_dir, exist_ok=True)
        print(f"\n=== Processing: {relative_path} ===")

        raw_text = extract_pdf_text(pdf_path)
        raw_length = len(raw_text)
        if not raw_text:
            print("  [WARN] No text extracted; skipping.")
            continue

        cleaned_text = ai_trim_whole_document(raw_text)
        kept_length = len(cleaned_text)
        percent_kept = 100.0 * kept_length / raw_length

        with open(output_path, "w", encoding="utf-8") as output_file:
            output_file.write(cleaned_text)

        records.append((relative_path, raw_length, kept_length, percent_kept))
        total_files += 1
        print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}% of characters)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")
for relative_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{relative_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

## KFUPM/Second Semester/KFUPM-2

### Process KFUPM/Second Semester/KFUPM-2

This block runs the processing pipeline for **KFUPM/Second Semester/KFUPM-2**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

## KFUPM/Second Semester/KFUPM-5

### Process KFUPM/Second Semester/KFUPM-5

This block runs the processing pipeline for **KFUPM/Second Semester/KFUPM-5**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

# KFUPM Coding Assignments

## Extract URL links from KFUPM files

### Extract URL links from KFUPM files

This block supports the **Extract URL links from KFUPM files** stage by extracting URLs or code artifacts from the KFUPM submission files and writing structured outputs that can be reviewed or reused later.

In [ ]:
%pip install --quiet pdfplumber python-docx

import html
import os
import re

import docx
import pdfplumber
from docx.opc.constants import RELATIONSHIP_TYPE
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    pass

PROJECT_DATA_DIR = os.environ.get("PROJECT_DATA_DIR")
SOURCE_DIRS = os.environ.get("URL_SOURCE_DIRS")

if not PROJECT_DATA_DIR:
    raise ValueError("Set PROJECT_DATA_DIR to the base directory containing source files.")
if not SOURCE_DIRS:
    raise ValueError(
        "Set URL_SOURCE_DIRS to source directories relative to PROJECT_DATA_DIR, "
        f"separated by {os.pathsep!r}."
    )

BASE = os.path.abspath(PROJECT_DATA_DIR)
TARGET_DIRS = [
    path if os.path.isabs(path) else os.path.join(BASE, path)
    for path in SOURCE_DIRS.split(os.pathsep)
    if path
]
OUTPUT_DOCX = os.environ.get(
    "URL_REPORT_DOCX", os.path.join(BASE, "extracted_urls.docx")
)

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)


def extract_text_from_pdf(path):
    try:
        with pdfplumber.open(path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    except Exception as exc:
        print(f"[PDF ERROR] {path}: {exc}")
        return ""


def extract_text_from_docx(path):
    try:
        document = docx.Document(path)
        return "\n".join(paragraph.text for paragraph in document.paragraphs)
    except Exception as exc:
        print(f"[DOCX ERROR] {path}: {exc}")
        return ""


def extract_text_from_txt(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as file:
            return file.read()
    except Exception as exc:
        print(f"[TXT ERROR] {path}: {exc}")
        return ""


def clean_url(raw):
    url = html.unescape(raw)

    for separator in ('"', "'", "<", " ", "\t", "\n"):
        if separator in url:
            url = url.split(separator)[0]

    url = url.rstrip(").,;:!?\"]'>&")
    return url if "." in url else ""


def extract_urls_from_text(text):
    urls = []
    seen = set()

    for match in URL_PATTERN.finditer(text):
        url = clean_url(match.group(0))
        if url and url not in seen:
            seen.add(url)
            urls.append(url)

    return urls


def add_hyperlink(paragraph, url, text=None):
    text = text or url
    relationship_id = paragraph.part.relate_to(
        url, RELATIONSHIP_TYPE.HYPERLINK, is_external=True
    )

    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), relationship_id)

    run = OxmlElement("w:r")
    properties = OxmlElement("w:rPr")

    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    properties.append(underline)

    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0000FF")
    properties.append(color)

    run.append(properties)
    text_element = OxmlElement("w:t")
    text_element.text = text
    run.append(text_element)

    hyperlink.append(run)
    paragraph._p.append(hyperlink)


document = docx.Document()
total_urls = 0

for folder in TARGET_DIRS:
    if not os.path.isdir(folder):
        print(f"[WARN] Folder not found: {folder}")
        continue

    heading = document.add_paragraph()
    heading.add_run(os.path.relpath(folder, BASE))
    heading.style = "Heading 1"
    document.add_paragraph("")

    for root, _, files in os.walk(folder):
        for filename in sorted(files):
            path = os.path.join(root, filename)
            extension = filename.lower()

            if extension.endswith(".pdf"):
                text = extract_text_from_pdf(path)
            elif extension.endswith(".docx"):
                text = extract_text_from_docx(path)
            elif extension.endswith(".txt"):
                text = extract_text_from_txt(path)
            else:
                continue

            for url in extract_urls_from_text(text):
                add_hyperlink(document.add_paragraph(), url)
                total_urls += 1

            document.add_paragraph("")

document.save(OUTPUT_DOCX)
print(f"Done. Wrote {total_urls} URLs into: {OUTPUT_DOCX}")

## Extract code from KFUPM URL links

### Extract code from KFUPM URL links

This block supports the **Extract code from KFUPM URL links** stage by extracting URLs or code artifacts from the KFUPM submission files and writing structured outputs that can be reviewed or reused later.

In [ ]:
#!/usr/bin/env python3
import hashlib
import json
import os
import re
import subprocess
import tempfile
from pathlib import Path
from urllib.parse import quote

PDF_WITH_URLS = os.environ.get("PDF_WITH_URLS")
URL_TEXT_FILE = os.environ.get("URL_TEXT_FILE")
DRIVE_OUT_ROOT = os.environ.get("DRIVE_OUT_ROOT", "output")

TOP_FILES_PER_URL = 30
MIN_LINES, MAX_LINES = 25, 2500
MAX_FILE_BYTES = 800_000

USE_OPENAI_TRIM = False
OPENAI_MODEL = "gpt-4.1"
OPENAI_MAX_CHARS_PER_FILE = 90_000

URL_RE = re.compile(r'https?://[^\s)"]+')

CODE_EXTS = {
    ".py",
    ".js",
    ".ts",
    ".jsx",
    ".tsx",
    ".java",
    ".kt",
    ".c",
    ".cpp",
    ".h",
    ".hpp",
    ".cs",
    ".go",
    ".rb",
    ".php",
    ".swift",
    ".rs",
    ".scala",
    ".sql",
    ".html",
    ".css",
    ".sass",
    ".scss",
    ".m",
    ".r",
    ".jl",
    ".sh",
    ".bash",
    ".zsh",
}

SKIP_DIRS = {
    ".git",
    "node_modules",
    "dist",
    "build",
    "out",
    ".next",
    ".nuxt",
    ".cache",
    ".venv",
    "venv",
    "env",
    "__pycache__",
    "vendor",
    "target",
    ".idea",
    ".vscode",
    ".pytest_cache",
    ".mypy_cache",
    "coverage",
    ".gradle",
}

SKIP_FILES = {
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
    "composer.lock",
    "poetry.lock",
    "Cargo.lock",
    "Gemfile.lock",
    "LICENSE",
    "LICENSE.md",
    "COPYING",
}

GEN_MARKERS = [
    "DO NOT EDIT",
    "Generated by",
    "This file was generated",
    "@generated",
    "AUTOGENERATED",
    "create-react-app",
    "Next.js",
    "Vite",
    "Angular CLI",
]


def run(cmd, cwd=None):
    process = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return process.returncode, process.stdout + "\n" + process.stderr


def normalize_url(url):
    return url.strip().rstrip(".,;")


def is_non_code_url(url):
    url_lower = url.lower()
    return any(
        marker in url_lower
        for marker in [
            "figma.com",
            "netlify.app",
            "vercel.app",
            "opengraph.githubassets.com",
        ]
    )


def is_code_url(url):
    url_lower = url.lower()
    return any(
        host in url_lower for host in ["github.com", "gitlab.com", "bitbucket.org"]
    )


def set_group_from_header(header):
    parts = [part.strip() for part in header.split("/") if part.strip()]
    if len(parts) < 3 or parts[0].upper() != "KFUPM":
        return None, None

    semester = parts[1]
    group = parts[2]
    if semester.lower().startswith("first"):
        semester = "First Semester"
    elif semester.lower().startswith("second"):
        semester = "Second Semester"
    if group.upper() == "KFUPM-4":
        group = "KFUPM-4"
    return semester, group


def read_lines_pdf(pdf_path):
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    lines = []
    for page in reader.pages:
        lines.extend((page.extract_text() or "").splitlines())
    return lines


def read_lines_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read().splitlines()


def parse_groups(lines):
    current_semester, current_group = None, None
    groups = {}
    current_submission = []

    def flush():
        nonlocal current_submission
        if current_semester and current_group and current_submission:
            groups.setdefault((current_semester, current_group), []).append(
                current_submission
            )
        current_submission = []

    for raw_line in lines:
        line = (raw_line or "").strip()
        if not line:
            flush()
            continue
        if line.startswith("KFUPM/"):
            flush()
            current_semester, current_group = set_group_from_header(line)
            continue
        current_submission.extend(normalize_url(url) for url in URL_RE.findall(line))

    flush()

    # PDF extraction may remove blank submission separators, so split multi-URL blocks.
    for group_key, submissions in list(groups.items()):
        fixed_submissions = []
        for submission in submissions:
            if len(submission) >= 2:
                fixed_submissions.extend([[url] for url in submission])
            else:
                fixed_submissions.append(submission)
        groups[group_key] = fixed_submissions

    return groups


def gh_parse(url):
    match = re.match(r"https?://github\.com/([^/]+)/([^/]+)(.*)$", url.strip())
    if not match:
        return None

    owner, repo, rest = (
        match.group(1),
        match.group(2).replace(".git", ""),
        match.group(3).strip("/"),
    )
    info = {
        "owner": owner,
        "repo": repo,
        "kind": "repo",
        "branch": None,
        "path": None,
        "file": None,
    }

    if rest.startswith("tree/"):
        info["kind"] = "tree"
        parts = rest.split("/", 2)
        info["branch"] = parts[1] if len(parts) > 1 else None
        info["path"] = parts[2] if len(parts) > 2 else ""
    elif rest.startswith("blob/"):
        info["kind"] = "blob"
        parts = rest.split("/", 2)
        info["branch"] = parts[1] if len(parts) > 1 else None
        info["file"] = parts[2] if len(parts) > 2 else ""

    return info


def git_clone(owner, repo, destination, branch=None):
    command = ["git", "clone", "--depth", "1"]
    if branch:
        command.extend(["--branch", branch])
    command.extend([f"https://github.com/{owner}/{repo}.git", destination])
    return run(command)[0] == 0


def gh_raw(owner, repo, branch, filepath):
    import requests

    branch = branch or "main"
    encoded_path = quote(filepath, safe="/")
    urls = [
        f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{encoded_path}",
        f"https://raw.githubusercontent.com/{owner}/{repo}/master/{encoded_path}",
    ]
    for raw_url in urls:
        response = requests.get(raw_url, timeout=30)
        if response.ok:
            return response.text
    return None


def keep_file(path):
    if path.name in SKIP_FILES or path.suffix.lower() not in CODE_EXTS:
        return False
    try:
        return path.stat().st_size <= MAX_FILE_BYTES
    except OSError:
        return False


def read_text(path):
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        try:
            return path.read_text(encoding="latin-1", errors="ignore")
        except OSError:
            return ""


def looks_minified(text):
    if len(text) < 2000:
        return False
    lines = text.splitlines()
    if not lines:
        return False
    return sum(len(line) > 400 for line in lines) / len(lines) > 0.35


def is_template(text):
    sample = text[:20000].lower()
    return any(marker.lower() in sample for marker in GEN_MARKERS)


def score(text):
    lines = text.splitlines()
    line_count = len(lines)
    if line_count < MIN_LINES or line_count > MAX_LINES or looks_minified(text):
        return 0

    nonempty = [line for line in lines if line.strip()]
    if not nonempty:
        return 0

    head = nonempty[: min(80, len(nonempty))]
    import_like = sum(
        bool(re.match(r"^\s*(import|from|#include|using|require\(|package\s)", line))
        for line in head
    )
    if import_like > 0.85 * len(head):
        return 1

    letters = sum(char.isalpha() for char in text[:8000])
    base = (line_count // 20) + (letters // 800)
    if is_template(text):
        base = max(1, base // 3)
    return max(1, base)


_client = None


def openai_trim(file_path, code_text):
    global _client

    if not USE_OPENAI_TRIM:
        return code_text, "OpenAI trim disabled"

    try:
        if _client is None:
            from openai import OpenAI

            _client = OpenAI()
    except Exception as error:
        return code_text, f"OpenAI init failed: {error}"

    chunk = code_text[:OPENAI_MAX_CHARS_PER_FILE]
    prompt = f"""Remove clearly templated, framework, or generated boilerplate while preserving student-authored logic.
Return JSON: {{"kept_code": "...", "removed_summary": "..."}}.
File: {file_path}
CODE:
```text
{chunk}
```"""

    try:
        response = _client.responses.create(model=OPENAI_MODEL, input=prompt)
        data = json.loads(getattr(response, "output_text", "") or "")
        return (
            data.get("kept_code", code_text),
            f"OpenAI removed: {data.get('removed_summary', 'n/a')}",
        )
    except Exception as error:
        return code_text, f"OpenAI trim failed: {error}"


def collect_from_github(url, tmpdir):
    items = []
    info = gh_parse(url)
    if not info:
        return items

    if info["kind"] == "blob":
        text = gh_raw(info["owner"], info["repo"], info["branch"], info["file"])
        if text and text.strip():
            file_score = score(text)
            if file_score > 0:
                items.append(
                    {
                        "url": url,
                        "file_rel": info["file"],
                        "text": text,
                        "score": file_score,
                    }
                )
        return items

    repo_dir = Path(tmpdir) / f"{info['owner']}__{info['repo']}"
    if not git_clone(info["owner"], info["repo"], str(repo_dir), info["branch"]):
        return items

    base = repo_dir
    if info["kind"] == "tree" and info["path"]:
        candidate = repo_dir / info["path"]
        if candidate.exists():
            base = candidate

    for root, directories, files in os.walk(base):
        directories[:] = [
            directory for directory in directories if directory not in SKIP_DIRS
        ]
        for filename in files:
            path = Path(root) / filename
            if not keep_file(path):
                continue
            text = read_text(path)
            if not text.strip():
                continue
            file_score = score(text)
            if file_score <= 0:
                continue
            items.append(
                {
                    "url": url,
                    "file_rel": str(path.relative_to(repo_dir)),
                    "text": text,
                    "score": file_score,
                }
            )

    return sorted(items, key=lambda item: item.get("score", 0), reverse=True)


def submission_hash(urls):
    return hashlib.sha1("\n".join(urls).encode("utf-8")).hexdigest()[:10]


def write_txt(output_path, urls, items, notes):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8", errors="ignore") as file:
        file.write("### SUBMISSION URLS\n" + "\n".join(urls) + "\n\n")
        file.write(
            "### NOTES\n"
            + "\n".join(f"- {note}" for note in (notes or ["(none)"]))
            + "\n\n"
        )
        if not items:
            file.write("### NO CODE EXTRACTED\n")
            return

        file.write("### EXTRACTED CODE (high-signal first)\n")
        for index, item in enumerate(items, 1):
            file.write("\n\n" + "=" * 90 + "\n")
            file.write(
                f"FILE {index}: {item.get('file_rel', '')}\nSOURCE URL: {item.get('url', '')}\n"
            )
            if "trim_note" in item:
                file.write(f"TRIM NOTE: {item['trim_note']}\n")
            file.write("=" * 90 + "\n\n")
            file.write(item.get("text", ""))
            if not item.get("text", "").endswith("\n"):
                file.write("\n")


def main():
    if not URL_TEXT_FILE and not PDF_WITH_URLS:
        raise ValueError("Set URL_TEXT_FILE or PDF_WITH_URLS in the environment.")

    lines = (
        read_lines_txt(URL_TEXT_FILE)
        if URL_TEXT_FILE
        else read_lines_pdf(PDF_WITH_URLS)
    )
    groups = parse_groups(lines)
    output_root = Path(DRIVE_OUT_ROOT)

    output_dirs = {
        ("First Semester", "KFUPM-3"): output_root / "First Semester" / "KFUPM-3",
        ("Second Semester", "KFUPM-3"): output_root / "Second Semester" / "KFUPM-3",
        ("Second Semester", "KFUPM-4"): output_root / "Second Semester" / "KFUPM-4",
    }
    for directory in output_dirs.values():
        directory.mkdir(parents=True, exist_ok=True)

    summary = []
    for group_key, submissions in groups.items():
        if group_key not in output_dirs:
            continue

        group_output = output_dirs[group_key]
        print(
            f"=== GROUP {group_key[0]}/{group_key[1]} | submissions: {len(submissions)} ==="
        )
        for index, urls in enumerate(submissions, 1):
            output_file = group_output / f"{index:03d}_{submission_hash(urls)}.txt"
            notes = []
            all_items = []

            with tempfile.TemporaryDirectory() as tmpdir:
                for url in urls:
                    if is_non_code_url(url):
                        notes.append(f"Skipped non-code URL: {url}")
                        continue
                    if not is_code_url(url):
                        notes.append(f"Skipped unsupported host: {url}")
                        continue
                    if "github.com" in url.lower():
                        items = collect_from_github(url, tmpdir)
                    else:
                        items = []
                        notes.append(f"Host handler not implemented: {url}")

                    if not items:
                        notes.append(
                            f"No code extracted (repository may be private or unavailable): {url}"
                        )
                        continue
                    all_items.extend(items[:TOP_FILES_PER_URL])

            seen = set()
            deduplicated = []
            for item in sorted(
                all_items, key=lambda value: value.get("score", 0), reverse=True
            ):
                key = (item.get("url", ""), item.get("file_rel", ""))
                if key not in seen:
                    seen.add(key)
                    deduplicated.append(item)

            if USE_OPENAI_TRIM:
                for item in deduplicated:
                    kept, note = openai_trim(
                        item.get("file_rel", ""), item.get("text", "")
                    )
                    item["text"] = kept
                    item["trim_note"] = note

            write_txt(output_file, urls, deduplicated, notes)
            summary.append(
                {
                    "group": f"{group_key[0]}/{group_key[1]}",
                    "submission_index": index,
                    "urls": urls,
                    "output": str(output_file),
                    "files_extracted": len(deduplicated),
                }
            )

    summary_path = output_root / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2, ensure_ascii=False)
    print(f"DONE. Output: {output_root}")
    print(f"Wrote: {summary_path}")


if __name__ == "__main__":
    main()

## Candidates

### KFUPM URL PDF -> logic-heavy candidate code extractor (GitHub-focused)

This block belongs to the **Candidates** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

In [ ]:
import hashlib
import json
import os
import re
import subprocess
import tempfile
from pathlib import Path

from pypdf import PdfReader

PDF_PATH = Path(os.environ.get("PDF_PATH", "input/urls.pdf"))
URL_TEXT_FILE_VALUE = os.environ.get("URL_TEXT_FILE")
URL_TEXT_FILE = Path(URL_TEXT_FILE_VALUE) if URL_TEXT_FILE_VALUE else None
OUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", "output"))

TOP_K_CANDIDATES = 12
FALLBACK_ONE_URL_PER_SUBMISSION = True
MAP_KFUPM4_TO_KFUPM5 = True

URL_RE = re.compile(r'https?://[^\s)"]+')


def normalize_url(url: str) -> str:
    return url.strip().rstrip(".,;")


def is_github_url(url: str) -> bool:
    return "github.com" in url.lower()


def gh_parse(url: str):
    match = re.match(r"https?://github\.com/([^/]+)/([^/]+)(.*)$", url.strip())
    if not match:
        return None

    owner = match.group(1)
    repo = match.group(2).replace(".git", "")
    rest = match.group(3).strip("/")
    info = {
        "owner": owner,
        "repo": repo,
        "kind": "repo",
        "branch": None,
        "path": None,
        "file": None,
    }

    if rest.startswith("tree/"):
        info["kind"] = "tree"
        parts = rest.split("/", 2)
        info["branch"] = parts[1] if len(parts) > 1 else None
        info["path"] = parts[2] if len(parts) > 2 else ""
    elif rest.startswith("blob/"):
        info["kind"] = "blob"
        parts = rest.split("/", 2)
        info["branch"] = parts[1] if len(parts) > 1 else None
        info["file"] = parts[2] if len(parts) > 2 else ""

    return info


def run(command, cwd=None):
    process = subprocess.run(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return process.returncode, f"{process.stdout}\n{process.stderr}"


def git_clone_depth1(
    owner: str, repo: str, dest_dir: Path, branch: str | None = None
) -> bool:
    dest_dir.parent.mkdir(parents=True, exist_ok=True)
    command = ["git", "clone", "--depth", "1"]
    if branch:
        command.extend(["--branch", branch])
    command.extend([f"https://github.com/{owner}/{repo}.git", str(dest_dir)])
    return run(command)[0] == 0


def read_lines_from_pdf(pdf_path: Path):
    reader = PdfReader(str(pdf_path))
    lines = []
    for page in reader.pages:
        lines.extend((page.extract_text() or "").splitlines())
    return lines


def read_lines_from_txt(txt_path: Path):
    return txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()


def set_group_from_header(header: str):
    parts = [part.strip() for part in header.split("/") if part.strip()]
    if len(parts) < 3 or parts[0].upper() != "KFUPM":
        return None, None

    semester = parts[1]
    group = parts[2].upper()
    if semester.lower().startswith("first"):
        semester = "First Semester"
    elif semester.lower().startswith("second"):
        semester = "Second Semester"

    if MAP_KFUPM4_TO_KFUPM5 and group == "KFUPM-4":
        group = "KFUPM-5"

    return semester, group


def parse_groups(lines):
    current_semester, current_group = None, None
    groups = {}
    current_submission = []

    def flush_submission():
        nonlocal current_submission
        if current_semester and current_group and current_submission:
            groups.setdefault((current_semester, current_group), []).append(
                current_submission
            )
        current_submission = []

    for raw_line in lines:
        line = (raw_line or "").strip()
        if not line:
            flush_submission()
            continue

        if line.startswith("KFUPM/"):
            flush_submission()
            current_semester, current_group = set_group_from_header(line)
            continue

        for url in URL_RE.findall(line):
            url = normalize_url(url)
            if is_github_url(url):
                current_submission.append(url)

    flush_submission()

    # PDF extraction may discard blank-line submission boundaries.
    if FALLBACK_ONE_URL_PER_SUBMISSION:
        for key, submissions in groups.items():
            groups[key] = [[url] for submission in submissions for url in submission]

    return groups


CODE_EXTS = {
    ".py",
    ".js",
    ".ts",
    ".jsx",
    ".tsx",
    ".java",
    ".kt",
    ".c",
    ".cpp",
    ".h",
    ".hpp",
    ".cs",
    ".go",
    ".rb",
    ".php",
    ".swift",
    ".rs",
    ".scala",
    ".sql",
    ".m",
    ".r",
    ".jl",
    ".sh",
    ".bash",
    ".zsh",
}

SKIP_DIRS = {
    ".git",
    "node_modules",
    "dist",
    "build",
    "out",
    ".next",
    ".nuxt",
    ".cache",
    ".venv",
    "venv",
    "env",
    "__pycache__",
    "vendor",
    "target",
    ".idea",
    ".vscode",
    ".pytest_cache",
    ".mypy_cache",
    "coverage",
    ".gradle",
}

SKIP_FILES = {
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
    "composer.lock",
    "poetry.lock",
    "Cargo.lock",
    "Gemfile.lock",
    "LICENSE",
    "LICENSE.md",
    "COPYING",
    "README.md",
    "readme.md",
}

GEN_NAME_PATTERNS = [
    r"\.gen\.",
    r"\.d\.ts$",
    r"__generated__",
    r"generated",
    r"routeTree\.gen\.",
]

TIER1 = [
    r"(^|/)(controllers?|services?|routes?|models?|entities?|dto|middleware|guards?|auth|security)(/|$)",
    r"(^|/)(db|database|schema|migrations?|prisma|typeorm|sequelize)(/|$)",
]
TIER2 = [
    r"(^|/)(server|backend|api)(/|$)",
    r"(^|/)(src)(/|$)",
    r"(^|/)(lib|utils?|helpers?|core)(/|$)",
]
PENALIZE = [r"(^|/)(public|assets|static|styles?|css)(/|$)"]


def is_generated_name(path: str) -> bool:
    normalized = path.lower().replace("\\", "/")
    return any(re.search(pattern, normalized) for pattern in GEN_NAME_PATTERNS)


def looks_minified(text: str) -> bool:
    if len(text) < 2000:
        return False
    lines = text.splitlines()
    if not lines:
        return False
    long_lines = sum(len(line) > 600 for line in lines)
    return long_lines / len(lines) > 0.25


def tier_bonus(path: str) -> int:
    normalized = path.lower().replace("\\", "/")
    score = 0
    if any(re.search(pattern, normalized) for pattern in TIER1):
        score += 80
    if any(re.search(pattern, normalized) for pattern in TIER2):
        score += 30
    if any(re.search(pattern, normalized) for pattern in PENALIZE):
        score -= 15
    return score


def token_score(text: str) -> int:
    score = 0
    score += 3 * len(
        re.findall(
            r"\b(if|else|elif|switch|case|for|while|try|catch|except|finally|throw)\b",
            text,
        )
    )
    score += 3 * len(
        re.findall(r"\b(function|def|class|interface|enum|async|await)\b", text)
    )
    score += 2 * len(re.findall(r"\b(return|yield|lambda)\b", text))
    score += 3 * len(
        re.findall(
            r"\b(fetch|axios|requests|router|get|post|put|delete)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += 3 * len(
        re.findall(
            r"\b(select|insert|update|join|where|schema|migration|prisma|typeorm|sequelize)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += 2 * len(
        re.findall(
            r"\b(validate|auth|token|jwt|bcrypt|hash|roles?|guard)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += len(
        re.findall(r"\b(error|exception|fail(ed|ure)?)\b", text, flags=re.IGNORECASE)
    )
    return score


def score_file(rel_path: str, code: str) -> int:
    rel_path = rel_path.replace("\\", "/")
    extension = Path(rel_path).suffix.lower()

    if extension not in CODE_EXTS:
        return 0
    if Path(rel_path).name in SKIP_FILES or is_generated_name(rel_path):
        return 0
    if not code.strip() or looks_minified(code):
        return 0

    lines = code.splitlines()
    if len(lines) < 25:
        return 0

    score = token_score(code) + tier_bonus(rel_path)
    nonempty_lines = [line for line in lines if line.strip()]
    head = nonempty_lines[:80]
    import_lines = sum(
        bool(re.match(r"^\s*(import|from|#include|using|require\(|package\s)", line))
        for line in head
    )
    if head and import_lines > 0.85 * len(head):
        score -= 25

    if extension in {".jsx", ".tsx", ".js", ".ts"}:
        jsx_tags = len(re.findall(r"<[A-Za-z]", code))
        control_flow = len(
            re.findall(r"\b(if|for|while|try|catch|switch|case)\b", code)
        )
        if jsx_tags > 250 and control_flow < 10:
            score -= 20

    return max(0, score)


def read_text(path: Path) -> str:
    for encoding in ("utf-8", "latin-1"):
        try:
            return path.read_text(encoding=encoding, errors="ignore")
        except OSError:
            continue
    return ""


def collect_candidates_from_repo(repo_dir: Path):
    candidates = []
    for root, directories, files in os.walk(repo_dir):
        directories[:] = [
            directory for directory in directories if directory not in SKIP_DIRS
        ]
        for filename in files:
            path = Path(root) / filename
            if path.suffix.lower() not in CODE_EXTS or path.name in SKIP_FILES:
                continue
            try:
                if path.stat().st_size > 1_000_000:
                    continue
            except OSError:
                continue

            code = read_text(path)
            if not code.strip():
                continue

            relative_path = str(path.relative_to(repo_dir)).replace("\\", "/")
            score = score_file(relative_path, code)
            if score > 0:
                candidates.append(
                    {"file_rel": relative_path, "score": score, "code": code}
                )

    return sorted(candidates, key=lambda candidate: candidate["score"], reverse=True)


def submission_id(urls):
    return hashlib.sha1("\n".join(urls).encode("utf-8")).hexdigest()[:10]


def write_candidates_txt(out_path: Path, urls, candidates, notes):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8", errors="ignore") as output:
        output.write("### SUBMISSION URLS\n")
        output.write("\n".join(urls) + "\n\n")
        output.write("### NOTES\n")
        output.write("\n".join(f"- {note}" for note in notes) if notes else "- (none)")
        output.write("\n")

        if not candidates:
            output.write("\n### NO CANDIDATES FOUND\n")
            return

        output.write("\n### LOGIC-HEAVY CANDIDATES (ranked)\n")
        for index, candidate in enumerate(candidates[:TOP_K_CANDIDATES], 1):
            output.write("\n" + "=" * 90 + "\n")
            output.write(f"CANDIDATE {index}\n")
            output.write(f"FILE: {candidate['file_rel']}\n")
            output.write(f"SCORE: {candidate['score']}\n")
            output.write("=" * 90 + "\n\n")
            output.write(candidate["code"])
            if not candidate["code"].endswith("\n"):
                output.write("\n")


def process_submission(urls, out_dir: Path):
    notes = []
    all_candidates = []

    with tempfile.TemporaryDirectory() as temporary_directory:
        temporary_directory = Path(temporary_directory)

        for index, url in enumerate(urls):
            url = normalize_url(url)
            if not is_github_url(url):
                notes.append(f"Skipped non-GitHub URL: {url}")
                continue

            info = gh_parse(url)
            if not info:
                notes.append(f"Could not parse GitHub URL: {url}")
                continue

            repo_dir = temporary_directory / f"{index}_{info['owner']}__{info['repo']}"
            if not git_clone_depth1(
                info["owner"], info["repo"], repo_dir, info["branch"]
            ):
                notes.append(f"Clone failed (private or unavailable): {url}")
                continue

            base = repo_dir
            if info["kind"] == "tree" and info["path"]:
                requested_path = repo_dir / info["path"]
                if requested_path.exists():
                    base = requested_path

            candidates = collect_candidates_from_repo(base)
            if not candidates:
                notes.append(f"No logic-heavy candidates found in repository: {url}")
                continue

            for candidate in candidates:
                candidate["source_url"] = url
            all_candidates.extend(candidates)

    best_by_file = {}
    for candidate in all_candidates:
        key = candidate["file_rel"]
        if key not in best_by_file or candidate["score"] > best_by_file[key]["score"]:
            best_by_file[key] = candidate

    merged = sorted(
        best_by_file.values(), key=lambda candidate: candidate["score"], reverse=True
    )
    out_path = out_dir / f"{submission_id(urls)}.txt"
    write_candidates_txt(out_path, urls, merged, notes)
    return out_path, len(merged)


def main():
    if URL_TEXT_FILE:
        lines = read_lines_from_txt(URL_TEXT_FILE)
    else:
        if not PDF_PATH.exists():
            raise FileNotFoundError(f"PDF not found: {PDF_PATH}")
        lines = read_lines_from_pdf(PDF_PATH)

    groups = parse_groups(lines)
    if not groups:
        all_urls = [
            normalize_url(url)
            for line in lines
            for url in URL_RE.findall(line)
            if is_github_url(normalize_url(url))
        ]
        submissions = (
            [[url] for url in all_urls]
            if FALLBACK_ONE_URL_PER_SUBMISSION
            else [all_urls]
        )
        groups = {("Ungrouped", "Ungrouped"): submissions}

    summary = []
    total = 0
    for (semester, group), submissions in groups.items():
        out_dir = OUT_ROOT / semester / group
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"=== {semester}/{group} | submissions: {len(submissions)} ===")
        for index, urls in enumerate(submissions, 1):
            out_path, candidate_count = process_submission(urls, out_dir)
            summary.append(
                {
                    "semester": semester,
                    "group": group,
                    "submission_index": index,
                    "urls": urls,
                    "output_txt": str(out_path),
                    "candidates_found": candidate_count,
                }
            )
            total += 1

    report = OUT_ROOT / "_summary.json"
    report.parent.mkdir(parents=True, exist_ok=True)
    report.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("Output folder:", OUT_ROOT)
    print("Total submissions processed:", total)
    print("Summary report:", report)


if __name__ == "__main__":
    main()

## Picking Candidates

### Picking Candidates

This block belongs to the **Picking Candidates** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

In [ ]:
import os
import re
from pathlib import Path

SRC_ROOT = Path(os.environ["CANDIDATES_SOURCE_DIR"]).expanduser()
DST_ROOT = Path(os.environ["TRIMMED_OUTPUT_DIR"]).expanduser()

MAX_TOTAL_LINES = 800
MAX_TOTAL_WORDS = 3000
MARK_LINE = "############"
IGNORE_IF_CANNOT_REACH = True
LOW_SIGNAL_SCORE_THRESHOLD = 220

CAND_HDR_RE = re.compile(r"^CANDIDATE\s+(\d+)\s*$", re.IGNORECASE)
FILE_RE = re.compile(r"^FILE:\s*(.+?)\s*$", re.IGNORECASE)
SCORE_RE = re.compile(r"^SCORE:\s*(\d+)\s*$", re.IGNORECASE)


def count_words(text: str) -> int:
    return len([word for word in text.replace("\u00a0", " ").split() if word.strip()])


def count_lines(text: str) -> int:
    return len(text.splitlines())


def parse_candidates(txt_path: Path):
    """Parse ranked candidate blocks from a generator output file."""
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    candidates = []
    current = None
    code_lines = []
    in_code = False

    def flush():
        nonlocal current, code_lines, in_code
        if current and current["file_rel"]:
            code = "\n".join(code_lines).rstrip()
            if code.strip():
                current["code"] = f"{code}\n"
                candidates.append(current)
        current = None
        code_lines = []
        in_code = False

    index = 0
    while index < len(lines):
        line = lines[index]
        candidate_match = CAND_HDR_RE.match(line.strip())

        if candidate_match:
            flush()
            current = {
                "cand_no": int(candidate_match.group(1)),
                "file_rel": "",
                "score": None,
                "code": "",
            }
            index += 1
            continue

        if current is None:
            index += 1
            continue

        file_match = FILE_RE.match(line.strip())
        if file_match and not in_code:
            current["file_rel"] = file_match.group(1).strip()
            index += 1
            continue

        score_match = SCORE_RE.match(line.strip())
        if score_match and not in_code:
            current["score"] = int(score_match.group(1))
            index += 1
            continue

        if not in_code:
            in_code = True
            if not line.strip() or set(line.strip()) == {"="}:
                index += 1
                continue

        if set(line.strip()) != {"="}:
            code_lines.append(line)
        index += 1

    flush()
    return candidates


# Favor implementation and data-layer paths; discount presentation-heavy paths.
TIER1 = [
    r"(^|/)(controllers?|services?|routes?|models?|entities?|dto|middleware|guards?|auth|security)(/|$)",
    r"(^|/)(db|database|schema|migrations?|prisma|typeorm|sequelize)(/|$)",
]
TIER2 = [
    r"(^|/)(server|backend|api)(/|$)",
    r"(^|/)(src)(/|$)",
    r"(^|/)(lib|utils?|helpers?|core)(/|$)",
]
PENALIZE = [
    r"(^|/)(public|assets|static|styles?|css)(/|$)",
    r"(^|/)(components?)(/|$)",
    r"(^|/)(pages?)(/|$)",
]


def tier_bonus(path: str) -> int:
    normalized_path = (path or "").lower().replace("\\", "/")
    bonus = 0
    if any(re.search(pattern, normalized_path) for pattern in TIER1):
        bonus += 80
    if any(re.search(pattern, normalized_path) for pattern in TIER2):
        bonus += 30
    if any(re.search(pattern, normalized_path) for pattern in PENALIZE):
        bonus -= 20
    return bonus


def token_score(text: str) -> int:
    score = 0
    score += 3 * len(
        re.findall(
            r"\b(if|else|elif|switch|case|for|while|try|catch|except|finally|throw)\b",
            text,
        )
    )
    score += 3 * len(
        re.findall(r"\b(function|def|class|interface|enum|async|await)\b", text)
    )
    score += 2 * len(re.findall(r"\b(return|yield|lambda)\b", text))
    score += 3 * len(
        re.findall(
            r"\b(fetch|axios|requests|router|get|post|put|delete)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += 3 * len(
        re.findall(
            r"\b(select|insert|update|join|where|schema|migration|prisma|typeorm|sequelize)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += 2 * len(
        re.findall(
            r"\b(validate|auth|token|jwt|bcrypt|hash|roles?|guard)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    score += len(
        re.findall(r"\b(error|exception|fail(ed|ure)?)\b", text, flags=re.IGNORECASE)
    )
    return score


def jsx_penalty(text: str) -> int:
    jsx_tags = len(re.findall(r"<[A-Za-z]", text))
    control_flow = len(re.findall(r"\b(if|for|while|try|catch|switch|case)\b", text))
    if jsx_tags > 400 and control_flow < 10:
        return 80
    if jsx_tags > 250 and control_flow < 10:
        return 40
    return 0


def score_output(selected_candidates) -> int:
    combined_code = "\n".join(candidate["code"] for candidate in selected_candidates)
    base_score = token_score(combined_code) - jsx_penalty(combined_code)
    path_bonus = sum(
        tier_bonus(candidate.get("file_rel", "")) for candidate in selected_candidates
    )
    return max(0, base_score + path_bonus)


def write_budget_file(dst_path: Path, selected_candidates) -> None:
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with dst_path.open("w", encoding="utf-8", errors="ignore") as output_file:
        for index, candidate in enumerate(selected_candidates):
            if index:
                output_file.write(f"\n{MARK_LINE}\n\n")
            output_file.write(f"{candidate['file_rel'].strip()}\n")
            code = candidate["code"]
            output_file.write(code if code.endswith("\n") else f"{code}\n")


def main() -> None:
    if not SRC_ROOT.exists():
        raise FileNotFoundError(f"Source directory not found: {SRC_ROOT}")

    DST_ROOT.mkdir(parents=True, exist_ok=True)

    processed = 0
    written = 0
    ignored = 0
    low_signal_outputs = []

    for root, _, files in os.walk(SRC_ROOT):
        root_path = Path(root)
        for filename in files:
            if not filename.lower().endswith(".txt"):
                continue

            src_path = root_path / filename
            relative_path = src_path.relative_to(SRC_ROOT)
            dst_path = (DST_ROOT / relative_path).with_suffix(".txt")
            candidates = parse_candidates(src_path)

            all_code = "\n".join(candidate["code"] for candidate in candidates)
            all_lines = count_lines(all_code)
            all_words = count_words(all_code)

            if (
                IGNORE_IF_CANNOT_REACH
                and all_lines < MAX_TOTAL_LINES
                and all_words < MAX_TOTAL_WORDS
            ):
                ignored += 1
                processed += 1
                continue

            selected = []
            total_lines = 0
            total_words = 0

            for candidate in candidates:
                if (
                    not candidate.get("code", "").strip()
                    or not candidate.get("file_rel", "").strip()
                ):
                    continue
                if total_lines >= MAX_TOTAL_LINES or total_words >= MAX_TOTAL_WORDS:
                    break

                code = candidate["code"]
                selected.append(candidate)
                total_lines += count_lines(code)
                total_words += count_words(code)

                if total_lines >= MAX_TOTAL_LINES or total_words >= MAX_TOTAL_WORDS:
                    break

            if total_lines < MAX_TOTAL_LINES and total_words < MAX_TOTAL_WORDS:
                ignored += 1
                processed += 1
                continue

            write_budget_file(dst_path, selected)
            written += 1
            processed += 1

            if score_output(selected) < LOW_SIGNAL_SCORE_THRESHOLD:
                low_signal_outputs.append(
                    str(dst_path.relative_to(DST_ROOT)).replace("\\", "/")
                )

    print("DONE")
    print("SRC:", SRC_ROOT)
    print("DST:", DST_ROOT)
    print("Processed:", processed)
    print("Written:", written)
    print("Ignored (could not reach 800 lines and 3000 words):", ignored)
    print("\nLOW-SIGNAL OUTPUT FILES (please check manually):")
    if not low_signal_outputs:
        print("(none)")
    else:
        for path in low_signal_outputs:
            print(path)


if __name__ == "__main__":
    main()

## File Types

### File Types

This block belongs to the **File Types** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

In [ ]:
drive.mount("/content/drive")

from collections import Counter

# ================== USER SETTINGS ==================
DIRS = {
    "First Semester / KFUPM-3": Path("${PROJECT_DATA_PATH}"),
    "Second Semester / KFUPM-4": Path("${PROJECT_DATA_PATH}"),
}

SEPARATOR_LINE = "############"
# ===================================================

LANG_MAP = {
    ".py": "Python",
    ".js": "JavaScript",
    ".jsx": "JavaScript (React JSX)",
    ".ts": "TypeScript",
    ".tsx": "TypeScript (React TSX)",
    ".java": "Java",
    ".kt": "Kotlin",
    ".c": "C",
    ".cpp": "C++",
    ".h": "C/C++ Header",
    ".hpp": "C++ Header",
    ".cs": "C#",
    ".go": "Go",
    ".rb": "Ruby",
    ".php": "PHP",
    ".swift": "Swift",
    ".rs": "Rust",
    ".scala": "Scala",
    ".sql": "SQL",
    ".m": "MATLAB/Objective-C",
    ".r": "R",
    ".jl": "Julia",
    ".sh": "Shell",
    ".bash": "Shell",
    ".zsh": "Shell",
    ".html": "HTML",
    ".css": "CSS",
    ".scss": "SCSS",
    ".sass": "Sass",
    ".json": "JSON",
    ".yml": "YAML",
    ".yaml": "YAML",
    ".xml": "XML",
    ".md": "Markdown",
}


def ext_to_lang(ext: str) -> str:
    ext = (ext or "").lower()
    return LANG_MAP.get(ext, f"Other ({ext})" if ext else "No extension")


def parse_file_names_from_txt(txt_path: Path):
    """
    Your format:
      <filename.ext>
      <code...>
      ############
      <filename2.ext>
      <code...>
      ############
      ...
    We count each filename line immediately after start or after a separator.
    """
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    names = []
    expecting_name = True  # first line is a filename

    for ln in lines:
        s = ln.strip()
        if s == SEPARATOR_LINE:
            expecting_name = True
            continue
        if expecting_name:
            if s:  # filename line
                names.append(s)
                expecting_name = False
            continue

    return names


def pct(n, total):
    return 100.0 * n / total if total else 0.0


def print_table(counter: Counter, title: str):
    total = sum(counter.values())
    rows = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    print(f"\n{title}")
    print(f"Total embedded code files counted: {total}")
    print("-" * 80)
    print(f"{'Language/Filetype':40s} | {'Count':>7s} | {'Percent':>8s}")
    print("-" * 80)
    for lang, cnt in rows:
        print(f"{lang:40s} | {cnt:7d} | {pct(cnt,total):7.2f}%")
    print("-" * 80)


def main():
    overall = Counter()
    per_folder = {}

    for label, folder in DIRS.items():
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")

        c = Counter()
        txt_files = 0

        for p in folder.rglob("*.txt"):
            txt_files += 1
            file_names = parse_file_names_from_txt(p)
            for name in file_names:
                ext = Path(name).suffix.lower()
                c[ext_to_lang(ext)] += 1

        per_folder[label] = (c, txt_files)
        overall.update(c)

    for label in DIRS:
        c, n_txt = per_folder[label]
        print_table(c, f"=== {label} (TXT files scanned: {n_txt}) ===")

    print_table(overall, "=== OVERALL (BOTH FOLDERS) ===")


main()

## Candidates

### KFUPM URL PDF -> logic-heavy candidate code extractor (GitHub-focused)

This block belongs to the **Candidates** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

## Picking Candidates

### Picking Candidates

This block belongs to the **Picking Candidates** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

## File Types

### File Types

This block belongs to the **File Types** stage of the coding-data workflow. It prepares candidate files for manual review, extracts useful metadata, or narrows the set of files that should move forward into the coding-analysis branch.

# Taif Assignments

## Taif/Medical/First Year/OSCE & Manuscripts

### Process Taif/Medical/First Year/OSCE & Manuscripts

This block runs the processing pipeline for **Taif/Medical/First Year/OSCE & Manuscripts**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re
from pathlib import Path

import pdfplumber
from openai import OpenAI

# Set this to True when running in Google Colab with Drive-backed data.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

# Configure these outside the notebook, for example:
# export PDF_ROOTS="/path/to/pdfs"
# export OUT_ROOT="/path/to/output"
PDF_ROOTS = [Path(path) for path in os.environ["PDF_ROOTS"].split(os.pathsep) if path]
OUT_ROOT = Path(os.environ["OUT_ROOT"])
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MAX_CHARS_PER_CHUNK = 6000
client = OpenAI()


def extract_pdf_text(pdf_path: Path) -> str:
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = (
                    (page.extract_text() or "")
                    .replace("\r\n", "\n")
                    .replace("\r", "\n")
                )
                parts.append(text)
    except Exception as error:
        print(f"[PDF ERROR] {pdf_path}: {error}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(
    text: str, max_chars: int = MAX_CHARS_PER_CHUNK
) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    instructions = """
Select and keep only text useful for AI-writing detection.

Keep verbatim:
- Complete paragraphs that appear to be student writing.
- Bulleted or numbered points with substantive content.
- Section titles associated with kept content.

Remove:
- Institutional boilerplate, cover pages, tables, figures, captions, equations,
  number-heavy lines, references, appendices, acknowledgements, page numbers,
  footers, and non-prose artifacts.

Do not paraphrase. Preserve order, join broken lines into paragraphs, retain blank
lines between paragraphs, and return only the cleaned text.
"""
    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            instructions=instructions,
            input=chunk_text,
            temperature=0.1,
        )
        return response.output_text
    except Exception as error:
        print(f"[API ERROR] {error}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)}...")
        cleaned = call_ai_trimmer(chunk).strip()
        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n\n".join(cleaned_parts)


total_files = 0
records = []

for pdf_root in PDF_ROOTS:
    print(f"\n===== SCANNING ROOT: {pdf_root} =====")

    for root, _, files in os.walk(pdf_root):
        for filename in files:
            if not filename.lower().endswith(".pdf"):
                continue

            pdf_path = Path(root) / filename
            relative_path = pdf_path.relative_to(pdf_root)
            output_dir = OUT_ROOT / pdf_root.name / relative_path.parent
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / f"{pdf_path.stem}_ai_trimmed.txt"

            print(f"\n=== Processing: {pdf_path} ===")
            raw_text = extract_pdf_text(pdf_path)
            raw_length = len(raw_text)

            if not raw_text:
                print("  [WARN] No text extracted; skipping.")
                continue

            cleaned_text = ai_trim_whole_document(raw_text)
            kept_length = len(cleaned_text)
            percent_kept = 100.0 * kept_length / raw_length

            output_path.write_text(cleaned_text, encoding="utf-8")
            records.append((pdf_path, raw_length, kept_length, percent_kept))
            total_files += 1

            print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}%)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")
for pdf_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{pdf_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

## Taif/Medical/Second Year

### Process Taif/Medical/Second Year

This block runs the processing pipeline for **Taif/Medical/Second Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re

import pdfplumber
from openai import OpenAI

PDF_ROOTS = [path for path in os.environ.get("PDF_ROOTS", "").split(os.pathsep) if path]
OUT_ROOT = os.environ.get("OUT_ROOT", "")
MAX_CHARS_PER_CHUNK = 6000

if not PDF_ROOTS:
    raise ValueError("Set PDF_ROOTS to one or more PDF directories.")
if not OUT_ROOT:
    raise ValueError("Set OUT_ROOT to an output directory.")

for pdf_root in PDF_ROOTS:
    if not os.path.isdir(pdf_root):
        raise ValueError(f"PDF root does not exist or is not a directory: {pdf_root}")

os.makedirs(OUT_ROOT, exist_ok=True)
client = OpenAI()


def extract_pdf_text(pdf_path: str) -> str:
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ""
                parts.append(text.replace("\r\n", "\n").replace("\r", "\n"))
    except Exception as exc:
        print(f"[PDF ERROR] {pdf_path}: {exc}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(text: str, max_chars: int = 6000) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    instructions = """
You are helping a researcher prepare student assignments for AI-writing detection.

From the input text, you must SELECT and KEEP ONLY the parts that are good for AI detection, and DELETE everything else.

KEEP (copy verbatim, do NOT rewrite):
- Complete paragraphs that look like real student writing.
- Bullet points and numbered points with real content.
- Section titles that belong to kept content.

DELETE:
- Headers, university boilerplate, cover pages.
- Tables, figures, captions.
- Equations and number-heavy lines.
- References, appendices, acknowledgements.
- Page numbers and footers.
- Any non-prose artifacts.

FORMAT:
- Never paraphrase.
- Preserve order.
- Join broken lines into full paragraphs.
- Keep blank lines between paragraphs.
- Output only cleaned text.
"""

    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            instructions=instructions,
            input=chunk_text,
            temperature=0.1,
        )
        return response.output_text
    except Exception as exc:
        print(f"[API ERROR] {exc}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text, MAX_CHARS_PER_CHUNK)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)}...")
        cleaned = call_ai_trimmer(chunk)
        if cleaned.strip():
            cleaned_parts.append(cleaned.strip())

    return "\n\n".join(cleaned_parts).strip()


total_files = 0
records = []

for pdf_root in PDF_ROOTS:
    print(f"\n===== SCANNING ROOT: {pdf_root} =====")
    root_name = os.path.basename(os.path.normpath(pdf_root))

    for root, _, files in os.walk(pdf_root):
        for filename in files:
            if not filename.lower().endswith(".pdf"):
                continue

            pdf_path = os.path.join(root, filename)
            relative_path = os.path.relpath(pdf_path, pdf_root)
            relative_dir = os.path.dirname(relative_path)
            base_name = os.path.splitext(filename)[0]

            output_dir = os.path.join(OUT_ROOT, root_name, relative_dir)
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"{base_name}_ai_trimmed.txt")

            print(f"\n=== Processing: {pdf_path} ===")
            raw_text = extract_pdf_text(pdf_path)
            raw_length = len(raw_text)

            if not raw_text:
                print("  [WARN] No text extracted, skipping.")
                continue

            cleaned_text = ai_trim_whole_document(raw_text)
            kept_length = len(cleaned_text)
            percent_kept = 100.0 * kept_length / raw_length

            with open(output_path, "w", encoding="utf-8") as output_file:
                output_file.write(cleaned_text)

            records.append((pdf_path, raw_length, kept_length, percent_kept))
            total_files += 1
            print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}%)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")

for pdf_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{pdf_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

## Taif/Mohammed/First Year

### Taif/Mohammed/First Year

This block runs the processing pipeline for **Taif/Mohammed/First Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

SRC_ROOT = Path(os.environ["SOURCE_ROOT"])
DST_ROOT = Path(os.environ["DESTINATION_ROOT"])
WORD_THRESHOLD = 50

subprocess.run(["bash", "-lc", "pip -q install pymupdf"], check=False)
subprocess.run(
    [
        "bash",
        "-lc",
        "apt-get -qq update && apt-get -qq install -y libreoffice >/dev/null",
    ],
    check=False,
)

import fitz


def count_words(text: str) -> int:
    # Tokenize on whitespace; preserves words in scripts like Arabic and ignores
    # non-word whitespace such as non-breaking spaces.
    return len([word for word in text.replace("\u00a0", " ").split() if word.strip()])


def first_page_word_count(pdf_path: Path) -> int:
    doc = fitz.open(str(pdf_path))
    if doc.page_count == 0:
        doc.close()
        return 0

    text = doc.load_page(0).get_text("text") or ""
    doc.close()
    return count_words(text)


def remove_first_page_if_short(
    pdf_in: Path,
    pdf_out: Path,
    threshold: int = WORD_THRESHOLD,
) -> tuple[bool, int]:
    word_count = first_page_word_count(pdf_in)
    pdf_out.parent.mkdir(parents=True, exist_ok=True)

    if word_count >= threshold:
        shutil.copy2(pdf_in, pdf_out)
        return False, word_count

    src = fitz.open(str(pdf_in))
    dst = fitz.open()
    for page_number in range(1, src.page_count):
        dst.insert_pdf(src, from_page=page_number, to_page=page_number)

    # PyMuPDF cannot save a zero-page document; create a blank page for one-page inputs.
    if dst.page_count == 0:
        dst.new_page()

    dst.save(str(pdf_out))
    dst.close()
    src.close()
    return True, word_count


def convert_docx_to_pdf(docx_path: Path, out_dir: Path) -> Path | None:
    out_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "libreoffice",
            "--headless",
            "--nologo",
            "--nofirststartwizard",
            "--convert-to",
            "pdf",
            "--outdir",
            str(out_dir),
            str(docx_path),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    pdf_path = out_dir / f"{docx_path.stem}.pdf"
    if pdf_path.exists():
        return pdf_path

    pdfs = list(out_dir.glob("*.pdf"))
    return pdfs[0] if pdfs else None


def rel_to_dst(src_path: Path) -> Path:
    return (DST_ROOT / src_path.relative_to(SRC_ROOT)).with_suffix(".pdf")


def process_pdf_file(src_pdf: Path) -> dict:
    dst_pdf = rel_to_dst(src_pdf)
    removed, word_count = remove_first_page_if_short(src_pdf, dst_pdf)
    return {
        "src": str(src_pdf),
        "dst": str(dst_pdf),
        "type": "pdf",
        "removed_first_page": removed,
        "first_page_words": word_count,
    }


def process_docx_file(src_docx: Path) -> dict:
    dst_pdf = rel_to_dst(src_docx)
    with tempfile.TemporaryDirectory() as tmp:
        converted = convert_docx_to_pdf(src_docx, Path(tmp))
        if not converted:
            return {
                "src": str(src_docx),
                "dst": str(dst_pdf),
                "type": "docx",
                "error": "DOCX-to-PDF conversion failed",
            }

        removed, word_count = remove_first_page_if_short(converted, dst_pdf)
        return {
            "src": str(src_docx),
            "dst": str(dst_pdf),
            "type": "docx",
            "removed_first_page": removed,
            "first_page_words": word_count,
        }


def main() -> None:
    if not SRC_ROOT.exists():
        raise FileNotFoundError(f"SOURCE_ROOT not found: {SRC_ROOT}")

    DST_ROOT.mkdir(parents=True, exist_ok=True)
    results = []
    n_docx = n_pdf = 0

    for root, _, files in os.walk(SRC_ROOT):
        root_path = Path(root)
        for filename in files:
            source_path = root_path / filename
            suffix = source_path.suffix.lower()

            if suffix == ".pdf":
                n_pdf += 1
                results.append(process_pdf_file(source_path))
            elif suffix == ".docx":
                n_docx += 1
                results.append(process_docx_file(source_path))

    successful = sum(1 for result in results if "error" not in result)
    errors = [result for result in results if "error" in result]
    removed = sum(1 for result in results if result.get("removed_first_page") is True)

    print("DONE")
    print(f"Source: {SRC_ROOT}")
    print(f"Destination: {DST_ROOT}")
    print(f"Found: {n_pdf} PDFs, {n_docx} DOCX")
    print(
        f"Processed OK: {successful} | Errors: {len(errors)} | "
        f"First page removed: {removed}"
    )

    if errors:
        print("\nErrors (first 20):")
        for error in errors[:20]:
            print("-", error["src"], "=>", error["error"])


main()

## Taif/Mohammed/Second Year

### Taif/Mohammed/Second Year

This block runs the processing pipeline for **Taif/Mohammed/Second Year**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

# Second Round AI Trim

### Processing round for Second Round AI Trim

This block runs the processing pipeline for **Second Round AI Trim**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
import os
import re

from openai import OpenAI

INPUT_ROOT = os.environ.get("AI_TRIM_INPUT_ROOT")
OUTPUT_ROOT = os.environ.get("AI_TRIM_OUTPUT_ROOT")

if not INPUT_ROOT or not OUTPUT_ROOT:
    raise EnvironmentError(
        "Set AI_TRIM_INPUT_ROOT and AI_TRIM_OUTPUT_ROOT before running this script."
    )
if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY before running this script.")

IN_ROOT = os.path.abspath(INPUT_ROOT)
OUT_ROOT = os.path.abspath(OUTPUT_ROOT)
if os.path.commonpath([IN_ROOT, OUT_ROOT]) == IN_ROOT:
    raise ValueError(
        "AI_TRIM_OUTPUT_ROOT must not be the input directory or a subdirectory of it."
    )

os.makedirs(OUT_ROOT, exist_ok=True)
client = OpenAI()

INSTRUCTIONS = """
You are cleaning extracted student-authored text for AI-detection.
You MUST NOT rewrite, paraphrase, summarize, translate, or add any words.
You may ONLY delete text and repair line breaks by joining lines.

KEEP ONLY:
- Complete student-authored prose paragraphs (full sentences).
- Coherent bullet/numbered lists that are complete.
- True section titles that belong to kept prose.

DELETE COMPLETELY:
- Any front-matter/boilerplate (university/course headers, templates, IDs, repeated headers).
- Tables/figures/captions and table-like text, reference lists, appendices, acknowledgements.
- Page numbers/footers and standalone numbers.
- Equation/symbol-heavy lines, artifacts, repeated junk, broken fragments.
- Any line that is clearly incomplete or a fragment.

PARAGRAPH REPAIR:
- Join broken lines inside a paragraph into one continuous paragraph.
- Keep blank lines between paragraphs.
- If a line does not end with sentence punctuation and looks cut off, delete it.

LIST RULES:
- Keep lists ONLY if you preserve their original order exactly.
- Do NOT renumber. Do NOT reorder.
- If a numbered list appears out of order OR missing numbers in the middle within a block, delete the ENTIRE list block.
- Sub-bullets like “- …”, “• …”, “o …” are allowed only if they clearly belong to the preceding item/section and are coherent.

OUTPUT SAFETY:
- Ensure the output does NOT end mid-word or mid-sentence.
- If the last line is incomplete, remove it.
- Output ONLY the cleaned text with no commentary.
""".strip()

MAX_CHARS = 3500


def chunk_text(text, max_chars=MAX_CHARS):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        candidate = f"{current}\n\n{block}".strip() if current else block
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = block
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


_SENT_END = re.compile(r'[.!?…]"?$')


def drop_trailing_incomplete_lines(text):
    lines = text.splitlines()
    while lines:
        last = lines[-1].strip()
        if not last:
            lines.pop()
            continue
        if (
            len(last) < 80
            and not _SENT_END.search(last)
            and not re.match(r"^[A-Z][A-Za-z0-9 \\-/():]{2,}$", last)
        ):
            lines.pop()
            continue
        break
    return "\n".join(lines).strip()


def remove_out_of_order_numbered_blocks(text):
    lines = text.splitlines()
    output = []
    index = 0

    while index < len(lines):
        match = re.match(r"^\s*(\d+)[\).\:-]\s+", lines[index])
        if not match:
            output.append(lines[index])
            index += 1
            continue

        block = []
        while index < len(lines):
            line = lines[index]
            if (
                re.match(r"^\s*(\d+)[\).\:-]\s+", line)
                or re.match(r"^\s*(?:[-•]|o)\s+", line)
                or not line.strip()
            ):
                block.append(line)
                index += 1
                continue
            break

        numbers = []
        for line in block:
            numbered = re.match(r"^\s*(\d+)[\).\:-]\s+", line)
            if numbered:
                numbers.append(int(numbered.group(1)))

        if all(
            numbers[position] == numbers[position - 1] + 1
            for position in range(1, len(numbers))
        ):
            output.extend(block)

    return "\n".join(output).strip()


def ai_trim_chunk(chunk):
    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=INSTRUCTIONS,
        input=chunk,
        temperature=0.0,
        max_output_tokens=4000,
    )
    return (response.output_text or "").strip()


def ai_trim_text(text):
    chunks = chunk_text(text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"   → chunk {index}/{len(chunks)}")
        cleaned = ai_trim_chunk(chunk)
        if cleaned:
            cleaned_parts.append(cleaned)

    merged = "\n\n".join(cleaned_parts).strip()
    merged = remove_out_of_order_numbered_blocks(merged)
    return drop_trailing_incomplete_lines(merged)


def pct(part, whole):
    return 0.0 if whole == 0 else 100.0 * part / whole


total = 0
summary = []

for root, _, files in os.walk(IN_ROOT):
    for filename in files:
        if not filename.lower().endswith(".txt"):
            continue

        input_path = os.path.join(root, filename)
        relative_path = os.path.relpath(input_path, IN_ROOT)
        output_path = os.path.join(OUT_ROOT, relative_path)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        print(f"\n=== Processing: {relative_path} ===")
        # Replacement decoding lets one malformed file continue through the batch.
        with open(input_path, "r", encoding="utf-8", errors="replace") as source:
            raw = source.read()

        raw_chars = len(raw)
        if not raw.strip():
            print("   [SKIP] empty")
            continue

        cleaned = ai_trim_text(raw)
        kept_chars = len(cleaned)
        kept_pct = pct(kept_chars, raw_chars)
        trimmed_pct = 100.0 - kept_pct

        with open(output_path, "w", encoding="utf-8") as destination:
            destination.write(cleaned)

        total += 1
        summary.append((relative_path, raw_chars, kept_chars, kept_pct, trimmed_pct))

        print(f"   Saved → {output_path}")
        print(
            f"   Kept: {kept_chars}/{raw_chars} chars ({kept_pct:.2f}%) | "
            f"Trimmed: {trimmed_pct:.2f}%"
        )

print("\n========== SUMMARY ==========")
print("Total processed:", total)

overall_original = sum(item[1] for item in summary)
overall_kept = sum(item[2] for item in summary)
overall_kept_pct = pct(overall_kept, overall_original)
overall_trimmed_pct = 100.0 - overall_kept_pct

print(
    f"OVERALL Kept: {overall_kept}/{overall_original} chars "
    f"({overall_kept_pct:.2f}%) | Trimmed: {overall_trimmed_pct:.2f}%\n"
)

for relative_path, original, kept, kept_pct, trimmed_pct in summary:
    print(
        f"{relative_path} | kept {kept_pct:.2f}% | trimmed {trimmed_pct:.2f}% | "
        f"{kept}/{original} chars"
    )

# Statistics

### Statistics

This block computes summary statistics or distribution checks for the **Statistics** stage. It helps verify corpus size, word-count ranges, and the effects of the trimming decisions made earlier in the notebook.

In [ ]:
drive.mount("/content/drive", force_remount=False)

from collections import defaultdict

ROOT = "${PROJECT_DATA_PATH}"
BIN = 600


def count_words(text: str) -> int:
    return len(re.findall(r"[\w\u0600-\u06FF]+(?:['-][\w\u0600-\u06FF]+)?", text))


def bin_label(w: int) -> str:
    if w < BIN:
        return f"<{BIN}"
    lo = (w // BIN) * BIN
    hi = lo + BIN
    return f"{lo}-{hi}"


file_rows = []  # (subfolder, relpath, words)
total_words_all = 0

for root, _, files in os.walk(ROOT):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        path = os.path.join(root, fn)
        rel = os.path.relpath(path, ROOT)
        subfolder = os.path.dirname(rel) if os.path.dirname(rel) else "."
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            txt = f.read()
        wc = count_words(txt)
        total_words_all += wc
        file_rows.append((subfolder, rel, wc))

if not file_rows:
    print("No .txt files found under:", ROOT)
else:
    # ---- files <300 ----
    lt300 = sorted(
        [r for r in file_rows if r[2] < 300], key=lambda x: (x[0], x[2], x[1])
    )
    print("FILES <300 WORDS (subfolder | words | file):")
    for sub, rel, wc in lt300:
        print(f"{sub} | {wc} | {rel}")

    # ---- per-subfolder bin distribution + per-subfolder word totals ----
    per_sub_counts = defaultdict(lambda: defaultdict(int))
    per_sub_files = defaultdict(int)
    per_sub_words = defaultdict(int)

    max_words = max(wc for _, _, wc in file_rows)
    labels = [f"<{BIN}"]
    hi = BIN
    while hi <= ((max_words // BIN) + 2) * BIN:
        labels.append(f"{hi}-{hi+BIN}")
        hi += BIN

    for sub, _, wc in file_rows:
        per_sub_files[sub] += 1
        per_sub_words[sub] += wc
        per_sub_counts[sub][bin_label(wc)] += 1

    print("\nPER-SUBFOLDER DISTRIBUTION (600-word bins):")
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        print(f"\n{sub}  (total files: {n} | total words: {per_sub_words[sub]})")
        for lab in labels:
            c = per_sub_counts[sub].get(lab, 0)
            pct = 100.0 * c / n
            if pct == 0:
                continue
            print(f"{lab}: {c} files ({pct:.2f}%)")

    # ---- word count per subfolder (requested) ----
    print(
        "\nWORD COUNT PER SUBFOLDER (subfolder | total words | files | avg words/file):"
    )
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        tw = per_sub_words[sub]
        avg = 0.0 if n == 0 else (tw / n)
        print(f"{sub} | {tw//1000}K | {n} | {(100*(avg//100)):.0f}")

    # ---- total words overall ----
    print("\nTOTAL WORDS ACROSS ALL FILES:", total_words_all)

### Statistics

This block computes summary statistics or distribution checks for the **Statistics** stage. It helps verify corpus size, word-count ranges, and the effects of the trimming decisions made earlier in the notebook.

In [ ]:
drive.mount('/content/drive', force_remount=False)

!pip install --quiet pdfplumber


ROOT = "${PROJECT_DATA_PATH}"
BIN = 600

# Counts "words" in English + Arabic text
def count_words(text: str) -> int:
    # Word = sequences of letters/numbers (includes Arabic letters), allowing internal ' or -
    tokens = re.findall(r"[A-Za-z0-9\u0600-\u06FF]+(?:['-][A-Za-z0-9\u0600-\u06FF]+)?", text)
    return len(tokens)

def read_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

def read_pdf_text(path: str) -> str:
    parts = []
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                t = page.extract_text() or ""
                t = t.replace("\r\n", "\n").replace("\r", "\n")
                parts.append(t)
    except Exception:
        return ""
    return "\n".join(parts)

def bin_label(w: int) -> str:
    if w < BIN:
        return f"<{BIN}"
    lo = (w // BIN) * BIN
    hi = lo + BIN
    return f"{lo}-{hi}"

# Collect: (subfolder, relpath, ext, words)
items = []
total_words_all = 0

for root, _, files in os.walk(ROOT):
    for fn in files:
        lower = fn.lower()
        if not (lower.endswith(".txt") or lower.endswith(".pdf")):
            continue

        path = os.path.join(root, fn)
        rel = os.path.relpath(path, ROOT)
        subfolder = os.path.dirname(rel) if os.path.dirname(rel) else "."
        ext = ".pdf" if lower.endswith(".pdf") else ".txt"

        if ext == ".txt":
            text = read_txt(path)
        else:
            text = read_pdf_text(path)

        wc = count_words(text)
        total_words_all += wc
        items.append((subfolder, rel, ext, wc))

if not items:
    print("No .txt or .pdf files found under:", ROOT)
else:
    # ---- files <300 words (includes txt + pdf) ----
    lt300 = sorted([x for x in items if x[3] < 300], key=lambda x: (x[0], x[3], x[1]))
    print("FILES <300 WORDS (subfolder | words | type | file):")
    for sub, rel, ext, wc in lt300:
        print(f"{sub} | {wc} | {ext} | {rel}")

    # ---- per-subfolder distribution + word totals ----
    per_sub_counts = defaultdict(lambda: defaultdict(int))
    per_sub_files  = defaultdict(int)
    per_sub_words  = defaultdict(int)

    max_words = max(wc for _, _, _, wc in items)
    labels = [f"<{BIN}"]
    hi = BIN
    while hi <= ((max_words // BIN) + 2) * BIN:
        labels.append(f"{hi}-{hi+BIN}")
        hi += BIN

    for sub, _, _, wc in items:
        per_sub_files[sub] += 1
        per_sub_words[sub] += wc
        per_sub_counts[sub][bin_label(wc)] += 1

    print("\nPER-SUBFOLDER DISTRIBUTION (600-word bins) — includes .txt and .pdf:")
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        tw = per_sub_words[sub]
        print(f"\n{sub}  (total files: {n} | total words: {tw})")
        for lab in labels:
            c = per_sub_counts[sub].get(lab, 0)
            pct = (100.0 * c / n)
            if pct == 0:
              continue
            print(f"{lab}: {c} files ({pct:.0f}%)")

    print("\nWORD COUNT PER SUBFOLDER (subfolder | total words | files | avg words/file):")
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        tw = per_sub_words[sub]
        avg = 0.0 if n == 0 else (tw / n)
        print(f"{sub} | {tw//1000}K | {n} | {100*(avg//100):.0f}")

    print("\nTOTAL WORDS ACROSS ALL FILES:", total_words_all)

### Statistics

This block computes summary statistics or distribution checks for the **Statistics** stage. It helps verify corpus size, word-count ranges, and the effects of the trimming decisions made earlier in the notebook.

In [ ]:
# items: iterable of tuples (subfolder, relpath, ext, word_count)
# Group files by subfolder, then sort each group's files by descending word count
# and ascending path for ties. Display average words per file rounded down to
# the nearest hundred and the top up to 20 files (word counts also floored to hundreds).
by_sub = defaultdict(list)
for sub, rel, ext, wc in items:
    by_sub[sub].append((wc, ext, rel))

for sub in sorted(by_sub.keys()):
    files = sorted(by_sub[sub], key=lambda x: (-x[0], x[2]))
    top_n = min(20, len(files))
    avg = sum(w for w, _, _ in files) / len(files) if files else 0.0

    print(f"\n{sub} | avg words/file: {(100*(avg//100)):.0f} | top {top_n} files:")
    for wc, ext, rel in files[:top_n]:
        print(f"{100*(wc//100)} | {ext} | {rel}")

# AI-Free Assignments (Pre-Fall 2022)

## First Round

### Process First Round

This block runs the processing pipeline for **First Round**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

In [ ]:
!pip install --quiet openai pdfplumber

import os
import re
from pathlib import Path

import pdfplumber
from openai import OpenAI

USE_GOOGLE_DRIVE = os.getenv("USE_GOOGLE_DRIVE", "0") == "1"
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(os.getenv("GOOGLE_DRIVE_MOUNT", "/content/drive"), force_remount=False)

PDF_ROOT = Path(os.getenv("PDF_ROOT", "data/pdfs"))
OUT_ROOT = Path(os.getenv("OUT_ROOT", "outputs/ai_trimmed"))
MAX_CHARS_PER_CHUNK = int(os.getenv("MAX_CHARS_PER_CHUNK", "6000"))
MAX_OUTPUT_TOKENS = int(os.getenv("MAX_OUTPUT_TOKENS", "16000"))

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
client = OpenAI()


def extract_pdf_text(pdf_path: Path) -> str:
    """Extract text page by page; scanned PDFs require separate OCR."""
    parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = (page.extract_text() or "").replace("\r\n", "\n").replace("\r", "\n")
                parts.append(text)
    except Exception as exc:
        print(f"[PDF ERROR] {pdf_path}: {exc}")
    return "\n\n".join(parts)


def split_into_chunks_by_paragraphs(text: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    """Keep paragraphs together where possible to retain local context."""
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = paragraph
        else:
            current = candidate

    if current:
        chunks.append(current)

    return chunks


def call_ai_trimmer(chunk_text: str) -> str:
    instructions = """
Select and keep only text suitable for AI-writing detection.

KEEP verbatim:
- Complete student-authored prose, including explanations, arguments, reflections, and descriptions.
- Bulleted or numbered points with substantive content.
- Headings associated with kept content.

DELETE:
- Institutional headers, templates, course metadata, student information, repeated boilerplate, and page footers.
- Tables of contents, indexes, lists of figures or tables, acknowledgements, copyright pages, appendices, references, and bibliographies.
- Table and figure captions or contents.
- Pure equations, formula-heavy lines, and lines dominated by numbers or symbols.
- Page numbers, PDF artifacts, and other non-prose material.

Do not paraphrase, summarize, or otherwise change retained wording. Preserve the original order. You may join broken lines within a paragraph. Keep headings on their own lines and separate paragraphs with blank lines. Output only the cleaned text.
"""
    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            instructions=instructions,
            input=chunk_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            temperature=0.1,
        )
        return response.output_text
    except Exception as exc:
        print(f"[API ERROR] {exc}")
        return ""


def ai_trim_whole_document(raw_text: str) -> str:
    chunks = split_into_chunks_by_paragraphs(raw_text)
    cleaned_parts = []

    for index, chunk in enumerate(chunks, start=1):
        print(f"  - Sending chunk {index}/{len(chunks)} to GPT-4.1-mini...")
        cleaned = call_ai_trimmer(chunk).strip()
        if cleaned:
            cleaned_parts.append(cleaned)

    return "\n\n".join(cleaned_parts)


total_files = 0
records = []

for root, _, files in os.walk(PDF_ROOT):
    for filename in files:
        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = Path(root) / filename
        relative_path = pdf_path.relative_to(PDF_ROOT)
        output_path = OUT_ROOT / relative_path.parent / f"{pdf_path.stem}_ai_trimmed.txt"
        output_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"\n=== Processing: {relative_path} ===")
        raw_text = extract_pdf_text(pdf_path)
        raw_length = len(raw_text)

        if not raw_text:
            print("  [WARN] No text extracted; skipping.")
            continue

        cleaned_text = ai_trim_whole_document(raw_text)
        kept_length = len(cleaned_text)
        percent_kept = 100.0 * kept_length / raw_length

        output_path.write_text(cleaned_text, encoding="utf-8")
        records.append((relative_path, raw_length, kept_length, percent_kept))
        total_files += 1
        print(f"  -> Saved: {output_path} (kept ~{percent_kept:.1f}% of characters)")

print("\n========== SUMMARY ==========")
print(f"Total PDFs processed: {total_files}")
for relative_path, raw_length, kept_length, percent_kept in records:
    print(
        f"{relative_path}: original {raw_length} chars, "
        f"kept {kept_length} (~{percent_kept:.1f}%)"
    )

11 files were not processed because of their .docx format. Those files were converted to .pdf, put in a different folder, and were processed using the same code and transferred to the same final destination.

### Process First Round

This block runs the processing pipeline for **First Round**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

## Second Round

### Processing round for Second Round

This block runs the processing pipeline for **Second Round**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

### Processing round for Second Round

This block runs the processing pipeline for **Second Round**. In general, these institution- or subgroup-specific blocks mount Drive, load the needed libraries, read the source files for that subgroup, apply the trimming or extraction rules, and write the cleaned outputs used later in the study.

## Removing Tables of Contents

### Strong TOC remover (prefix short-lines until first real paragraph)

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

In [ ]:
import os
import re

INPUT_ROOT = os.environ.get("TOC_INPUT_ROOT")
OUTPUT_ROOT = os.environ.get("TOC_OUTPUT_ROOT")

if not INPUT_ROOT or not OUTPUT_ROOT:
    raise ValueError(
        "Set TOC_INPUT_ROOT and TOC_OUTPUT_ROOT to input and output directories."
    )

INPUT_ROOT = os.path.abspath(INPUT_ROOT)
OUTPUT_ROOT = os.path.abspath(OUTPUT_ROOT)
if INPUT_ROOT == OUTPUT_ROOT:
    raise ValueError(
        "TOC_OUTPUT_ROOT must differ from TOC_INPUT_ROOT to avoid overwriting source files."
    )

os.makedirs(OUTPUT_ROOT, exist_ok=True)

SCAN_MAX_LINES = 500
SHORT_MAX_CHARS = 85
PARA_MIN_CHARS = 140
PARA_MIN_WORDS = 22
ALLOW_SHORT_BLANKS = True
MAX_LEADING_BLANKS = 50


def is_paragraph_line(line: str) -> bool:
    text = line.strip()
    if not text:
        return False

    words = re.findall(r"[A-Za-z]+", text)
    if sum(len(word) for word in words) < 20:
        return False

    return len(text) >= PARA_MIN_CHARS or len(words) >= PARA_MIN_WORDS


def is_shortish(line: str) -> bool:
    text = line.strip()
    if not text:
        return ALLOW_SHORT_BLANKS
    return len(text) <= SHORT_MAX_CHARS


def detect_cut_index(lines: list[str]) -> int:
    start = 0
    while (
        start < len(lines) and start < MAX_LEADING_BLANKS and not lines[start].strip()
    ):
        start += 1

    scan_end = min(len(lines), SCAN_MAX_LINES)
    if start < scan_end and is_paragraph_line(lines[start]):
        return 0

    for index in range(start, scan_end):
        if is_paragraph_line(lines[index]):
            return index
        if not is_shortish(lines[index]):
            continue

    return 0


def remove_toc_prefix_keep_lastline(text: str) -> tuple[str, bool]:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.splitlines()

    cut = detect_cut_index(lines)
    if cut <= 0:
        return text, False

    removed = lines[:cut]
    kept = lines[cut:]

    # Retain the final removed line because it commonly identifies the first section.
    last_title = next(
        (line.rstrip() for line in reversed(removed) if line.strip()),
        "",
    )

    first_kept_nonempty = next((line.strip() for line in kept if line.strip()), "")
    if first_kept_nonempty == last_title.strip():
        last_title = ""

    output_lines = []
    if last_title:
        output_lines.extend((last_title, ""))
    output_lines.extend(kept)

    return "\n".join(output_lines).lstrip("\n"), True


def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="replace") as file:
        return file.read()


def write_text(path: str, text: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as file:
        file.write(text)


total = 0
changed = 0

for root, _, files in os.walk(INPUT_ROOT):
    for filename in files:
        if not filename.lower().endswith(".txt"):
            continue

        input_path = os.path.join(root, filename)
        relative_path = os.path.relpath(input_path, INPUT_ROOT)
        output_path = os.path.join(OUTPUT_ROOT, relative_path)

        cleaned, did_change = remove_toc_prefix_keep_lastline(read_text(input_path))
        write_text(output_path, cleaned)

        total += 1
        changed += did_change

print("Total text files:", total)
print("Files with removed prefixes:", changed)
print("Output directory:", OUTPUT_ROOT)

## Removing Short and Long Files & Word Count

### Removing Short and Long Files & Word Count

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

In [ ]:
drive.mount("/content/drive", force_remount=False)


ROOT = "${PROJECT_DATA_PATH}"
BIN = 600
DELETE_OVER = 6600  # delete files with more than this many words


def count_words(text: str) -> int:
    return len(re.findall(r"[\w\u0600-\u06FF]+(?:['-][\w\u0600-\u06FF]+)?", text))


def bin_label(w: int) -> str:
    if w < BIN:
        return f"<{BIN}"
    lo = (w // BIN) * BIN
    hi = lo + BIN
    return f"{lo}-{hi}"


file_rows = []  # (subfolder, relpath, words, fullpath)
total_words_all = 0

for root, _, files in os.walk(ROOT):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        path = os.path.join(root, fn)
        rel = os.path.relpath(path, ROOT)
        subfolder = os.path.dirname(rel) if os.path.dirname(rel) else "."
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            txt = f.read()
        wc = count_words(txt)
        total_words_all += wc
        file_rows.append((subfolder, rel, wc, path))

if not file_rows:
    print("No .txt files found under:", ROOT)
else:
    # ---- delete files > 6600 words ----
    to_delete = sorted(
        [r for r in file_rows if r[2] > DELETE_OVER], key=lambda x: (x[0], x[2], x[1])
    )
    print(f"FILES > {DELETE_OVER} WORDS TO DELETE:")
    for sub, rel, wc, _ in to_delete:
        print(f"{sub} | {wc} | {rel}")

    deleted_count = 0
    deleted_words = 0
    for sub, rel, wc, path in to_delete:
        try:
            os.remove(path)
            deleted_count += 1
            deleted_words += wc
        except Exception as e:
            print(f"Could not delete {rel}: {e}")

    print(f"\nDELETED {deleted_count} FILE(S) WITH > {DELETE_OVER} WORDS.")
    print(f"REMOVED WORDS FROM TOTAL: {deleted_words}")

    # keep only remaining files for stats
    file_rows = [r for r in file_rows if r[2] <= DELETE_OVER]
    total_words_all -= deleted_words

    if not file_rows:
        print("\nNo files remain after deletion.")
    else:
        # ---- files <300 ----
        lt300 = sorted(
            [r for r in file_rows if r[2] < 300], key=lambda x: (x[0], x[2], x[1])
        )
        print("\nFILES <300 WORDS (subfolder | words | file):")
        for sub, rel, wc, _ in lt300:
            print(f"{sub} | {wc} | {rel}")

        # ---- per-subfolder bin distribution + per-subfolder word totals ----
        per_sub_counts = defaultdict(lambda: defaultdict(int))
        per_sub_files = defaultdict(int)
        per_sub_words = defaultdict(int)

        max_words = max(wc for _, _, wc, _ in file_rows)
        labels = [f"<{BIN}"]
        hi = BIN
        while hi <= ((max_words // BIN) + 2) * BIN:
            labels.append(f"{hi}-{hi+BIN}")
            hi += BIN

        for sub, _, wc, _ in file_rows:
            per_sub_files[sub] += 1
            per_sub_words[sub] += wc
            per_sub_counts[sub][bin_label(wc)] += 1

        print("\nPER-SUBFOLDER DISTRIBUTION (600-word bins):")
        for sub in sorted(per_sub_files.keys()):
            n = per_sub_files[sub]
            print(f"\n{sub}  (total files: {n} | total words: {per_sub_words[sub]})")
            for lab in labels:
                c = per_sub_counts[sub].get(lab, 0)
                pct = 100.0 * c / n
                if pct == 0:
                    continue
                print(f"{lab}: {c} files ({pct:.2f}%)")

        # ---- word count per subfolder ----
        print(
            "\nWORD COUNT PER SUBFOLDER (subfolder | total words | files | avg words/file):"
        )
        for sub in sorted(per_sub_files.keys()):
            n = per_sub_files[sub]
            tw = per_sub_words[sub]
            avg = 0.0 if n == 0 else (tw / n)
            print(f"{sub} | {tw//1000}K | {n} | {(100*(avg//100)):.0f}")

        # ---- total words overall ----
        print("\nTOTAL WORDS ACROSS ALL REMAINING FILES:", total_words_all)

# Resolving Issues & Filling Gaps

## Removal of Table of Contents from Beginnings

### Strong TOC remover (prefix short-lines until first real paragraph)

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

## Removal of short files

### Removal of short files

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

In [ ]:
drive.mount("/content/drive", force_remount=False)

import os
import shutil

ROOT = "${PROJECT_DATA_PATH}"
QUARANTINE = "${PROJECT_DATA_PATH}"
os.makedirs(QUARANTINE, exist_ok=True)

# Assumes you already computed lt300 earlier as a list of tuples like:
# lt300 = [(subfolder, relpath, ext, words), ...]   OR   [(subfolder, relpath, words), ...]
# If lt300 doesn't exist in your runtime, re-run your counting cell first.

moved = 0
missing = 0

for row in lt300:
    # Support both formats
    if len(row) == 4:
        sub, rel, ext, wc = row
    else:
        sub, rel, wc = row
        ext = os.path.splitext(rel)[1].lower()

    src = os.path.join(ROOT, rel)
    dst = os.path.join(QUARANTINE, rel)

    if not os.path.exists(src):
        missing += 1
        continue

    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(src, dst)
    moved += 1

print("DONE")
print("Moved out of final folder (<300 words):", moved)
print("Missing (not found at expected path):", missing)
print("Moved files location:", QUARANTINE)

## Limiting size of large subfolders

### Limiting size of large subfolders

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

In [ ]:
drive.mount("/content/drive", force_remount=False)

import os
import shutil
import random

ROOT = "${PROJECT_DATA_PATH}"
TARGET_SUBFOLDER = "Taif/Mohammed/PT"  # relative to ROOT
KEEP_N = 52
MIN_WORDS = 600

QUARANTINE = f"{ROOT} - Removed_Others_{TARGET_SUBFOLDER.replace('/','_')}"
os.makedirs(QUARANTINE, exist_ok=True)

# Assumes you already have in memory:
# items = [(subfolder, relpath, ext, word_count), ...]
# from the earlier counting cell.

eligible = [
    rel for sub, rel, ext, wc in items if sub == TARGET_SUBFOLDER and wc > MIN_WORDS
]

if len(eligible) == 0:
    raise ValueError(f"No files with >{MIN_WORDS} words in {TARGET_SUBFOLDER}")

random.seed()  # set a fixed int for reproducibility if you want
keep = set(random.sample(eligible, k=min(KEEP_N, len(eligible))))

# Move out everything else in that subfolder
moved = 0
kept = 0
missing = 0

target_abs = os.path.join(ROOT, TARGET_SUBFOLDER)
if not os.path.exists(target_abs):
    raise FileNotFoundError(f"Target folder not found: {target_abs}")

for root, _, files in os.walk(target_abs):
    for fn in files:
        if not (fn.lower().endswith(".txt") or fn.lower().endswith(".pdf")):
            continue

        src = os.path.join(root, fn)
        rel = os.path.relpath(src, ROOT)

        if rel in keep:
            kept += 1
            continue

        dst = os.path.join(QUARANTINE, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(src):
            shutil.move(src, dst)
            moved += 1
        else:
            missing += 1

print("DONE")
print("Target subfolder:", TARGET_SUBFOLDER)
print(f"Eligible (> {MIN_WORDS} words):", len(eligible))
print("Kept (random):", kept)
print("Removed (moved to quarantine):", moved)
print("Missing:", missing)
print("Quarantine folder:", QUARANTINE)

### Uses stored variables only

This block performs the cleanup task named above for the current sample. It resolves a specific data-quality issue so the final trimmed corpus is more consistent and easier to analyze.

In [ ]:
# Exclude documents below the word threshold and unselected documents
# from the configured sampled subfolder.
removed_lt300 = {row[1] for row in lt300}
removed_sampled = {
    rel for sub, rel, _, _ in items if sub == TARGET_SUBFOLDER and rel not in keep
}
removed_all = removed_lt300 | removed_sampled

old_total_files = len(items)
old_total_words = sum(wc for _, _, _, wc in items)

new_items = [
    (sub, rel, ext, wc) for sub, rel, ext, wc in items if rel not in removed_all
]

new_total_files = len(new_items)
new_total_words = sum(wc for _, _, _, wc in new_items)
removed_words = sum(wc for _, rel, _, wc in items if rel in removed_all)

print("OLD totals:")
print("files:", old_total_files)
print("words:", old_total_words)

print("\nREMOVED totals:")
print("removed files:", len(removed_all))
print("removed words:", removed_words)

print("\nNEW totals (after threshold and subfolder filtering):")
print("files:", new_total_files)
print("words:", new_total_words)

# Statistics

### Statistics

This block computes summary statistics or distribution checks for the **Statistics** stage. It helps verify corpus size, word-count ranges, and the effects of the trimming decisions made earlier in the notebook.